In [ ]:
%load_ext autoreload
%autoreload 2

import os
import sys
from pathlib import Path

notebook_dir = Path.cwd()
root_dir = notebook_dir.parent
sys.path.insert(0, str(root_dir))

import numpy as np
import pandas as pd
import random

from COMIND_transformer.visualization import *
from COMIND_transformer.utils import *

from COMIND_transformer.em_transformer import EM

import statsmodels.formula.api as smf
from sklearn.model_selection import KFold, GridSearchCV

### Data loading

In [2]:
df = pd.read_csv("/data01/bgutman/MRI_data/PPMI/data_ppmi_pd.csv")
df_K = pd.read_csv("/data01/bgutman/LEGACY/Skoltech/datasets/Connectomes/mean_NORM_con_22.csv")
#df_K = pd.read_csv("/data01/bgutman/LEGACY/Skoltech/datasets/Connectomes/mean_NORM_con_min_edges=3__sparsity=0.3.csv")

### Cleaning data
This is to remove NaNs and infs, as well as removing single timepoint observations.

In [3]:
## remove non-longitudinal observations
print("original size:", df.shape)
relevant_cols = [col for col in df.columns if col.startswith(('L_', 'R_')) and ('_thickavg' in col or '_thickavg_resid' in col)]
relevant_cols += ["MCATOT", "TD_score", "PIGD_score"]
df = df.replace([np.inf, -np.inf], np.nan)
df = df.dropna(subset=relevant_cols)

print("after drop na", df.shape)
subj_counts = df['subj_id'].value_counts()
num_unique = (subj_counts == 1).sum()
print("one time subj_id:", num_unique)

longitudinal_ids = subj_counts[subj_counts > 1].index
df = df[df['subj_id'].isin(longitudinal_ids)].copy()
df = df.drop_duplicates(subset=["subj_id", "time"])
print("after drop dupes", df.shape)

X_obs = df[[col for col in df.columns if (col.startswith(('L_', 'R_')) and col.endswith('_thickavg') and not col.endswith('_thickavg_resid'))]]

biomarker_names = [col for col in df.columns 
                   if col.startswith(('L_', 'R_')) and 
                   col.endswith('_thickavg') and 
                   not col.endswith('_thickavg_resid')]

#print(biomarker_names)

# X_obs = df[small_region_set]
X_obs = X_obs.to_numpy()
X_obs = np.max(X_obs, axis=0) - X_obs

print("nans in X:", np.isnan(X_obs).sum())
print("infs in X:", np.isinf(X_obs).sum())

original size: (880, 250)
after drop na (868, 250)
one time subj_id: 227
after drop dupes (504, 250)
nans in X: 0
infs in X: 0


#### Zeroing diagonal of transition matrix + normalization

In [4]:
## connectivity matrix to numpy
K = df_K.drop(df_K.columns[0], axis=1).to_numpy()
np.fill_diagonal(K, 0)
print(K.shape, type(K))

# normalization
row_sums = K.sum(axis=1)
median_row_sum = np.median(row_sums)
K = K / median_row_sum

(68, 68) <class 'numpy.ndarray'>


#### creating extracting data for transformer
subjid, time between observations, clinical scores (under "`cog`" variable), HY score, and NSD score.

HY and NSD are used for validation, not in the transformer.

In [5]:
ids = df["subj_id"].to_numpy()
dt = df["time"].to_numpy()/12 # convert to years
cog = df[["MCATOT","TD_score","PIGD_score"]].to_numpy()
nhy = df["NHY"].to_numpy()
# print("nans in cog:", np.isnan(cog).sum())
# print("infs in cog:", np.isinf(cog).sum())

df["NSD_STAGE"] = df["NSD_STAGE"].replace({"Not NSD": 0, "2b": 2})
df["NSD_STAGE"] = pd.to_numeric(df["NSD_STAGE"], errors='coerce')  # handles any remaining invalid entries

patient_df = df.copy()
patient_df["NHY"] = nhy  # add NHY array to df
grouped = patient_df.groupby("subj_id")[["NSD_STAGE", "NHY"]].mean().dropna()

nsd = df["NSD_STAGE"].to_numpy()

In [ ]:
from COMIND_transformer.features.beta_glm import fit_mixedlm_beta_from_clinical

t_max = 40
n_biomarkers = 68

initial_beta, pid_to_beta, _ = fit_mixedlm_beta_from_clinical(df=df, 
                                                              ids=ids,
                                                              dt=dt, 
                                                              t_max=t_max, 
                                                              verbose=True
                                                              )

from sklearn.model_selection import train_test_split
X = create_patient_list(X_obs, ids, dt, cog, nsd, initial_beta)
X_train, X_val = train_test_split(X, test_size=0.2, random_state=75)

# Save train and validation data
out_dir = "/home/dsemchin/COMIND/results"
os.makedirs(out_dir, exist_ok=True)

np.save(os.path.join(out_dir, "X_train.npy"), X_train)
np.save(os.path.join(out_dir, "X_val.npy"), X_val)
print(f"Saved training data: {os.path.join(out_dir, 'X_train.npy')}")
print(f"Saved validation data: {os.path.join(out_dir, 'X_val.npy')}")



beta_init summary: count    146.000000
mean      15.622561
std        7.789749
min        0.000000
25%        9.586120
50%       15.547792
75%       20.682332
max       40.000000
dtype: float64


#### example patient in X

In [7]:
print(X[0].keys())
print(X[0]["X_obs"].shape)
print(X[0]["dt"].shape)
print(X[0]["cog"].shape)
print(X[0]["nhy"].shape)
print(X[0]["initial_beta"])

dict_keys(['id', 'X_obs', 'dt', 'cog', 'nhy', 'initial_beta'])
(2, 68)
(2,)
(2, 3)
(2,)
24.50505528293517


### Single fit example

In [ ]:

# Initialize forcing term f using eigenvector based initialization + jitter
f_init_list = initialize_f_eigen(K=K, 
                                 jitter_strength=0.05, 
                                 n_eigs=10,
                                 rng=np.random.RandomState(75)
                                 )
f_init = f_init_list[0]

# Create and fit the model
em = EM(K=K,
        lambda_f=1.0,
        lambda_cog=0.01,
        lambda_scalar=0.3,
        initial_f=f_init,
        jac_toggle=True,
        max_iter=8,
        t_max=40,
        epsilon=1e-1)

em.fit(X_train)
beta_val = em.transform(X_val)

# Save results
out_dir = "/home/dsemchin/COMIND/results"
os.makedirs(out_dir, exist_ok=True)
out_path = os.path.join(out_dir, "ppmi_single_fit.npz")

np.savez(out_path,
         theta_history=np.array(em.theta_history),
         cog_history=np.array(em.cog_regression_history),
         beta_history=np.array(em.beta_history),
         lse_history=np.array(em.lse_history),
         beta_val=np.array(beta_val),
         f_init=f_init)
print("Saved:", out_path)

Estimating beta values: 100%|██████████| 30/30 [00:00<00:00, 46.30it/s]

Saved: /home/dsemchin/COMIND/results/ppmi_single_fit.npz


### KFold GridSearch

In [10]:
# Initialize F for grid search (using first eigenvector initialization)
f_init_grid = initialize_f_eigen(K=K, 
                                  jitter_strength=0.05, 
                                  n_eigs=1,
                                  rng=np.random.RandomState(75)
                                  )[0]

param_grid = {
    "lambda_f": [0.5, 0.9],#[0.9, 0.95, 1.0, 1.1, 1.2],
    "lambda_cog": [1e-2, 1e-3],
    "lambda_scalar": [0.3], #[0.1, 0.3, 0.5, 0.7, 0.9],
    "jac_toggle": [True],
    "max_iter": [100],
    "t_max": [40],
    "epsilon": [1e-2],
    "initial_f": [f_init_grid],
}

# Use regular KFold
grid = GridSearchCV(
    estimator=EM(K=K),
    param_grid=param_grid,
    cv=KFold(n_splits=3, shuffle=True, random_state=75),
    scoring=None,
    n_jobs=28
)
grid.fit(X=X_train, y=None)

print("Best score:", grid.best_score_)
print("Best params:", grid.best_params_)

# Transform validation set with best model
best_model = grid.best_estimator_
beta_val = best_model.transform(X_val)

# Save grid search results
out_path = os.path.join(out_dir, "ppmi_gridsearch_results.npz")

np.savez(out_path,
         theta_history=np.array(best_model.theta_history),
         cog_history=np.array(best_model.cog_regression_history),
         beta_history=np.array(best_model.beta_history),
         lse_history=np.array(best_model.lse_history),
         beta_val=np.array(beta_val),
         best_params=grid.best_params_,
         cv_results=grid.cv_results_)
print("Saved:", out_path)


100%|██████████| 1/1 [00:29<00:00, 29.86s/it]


Best score: -311.6731120028631
Best params: {'epsilon': 0.01, 'initial_f': array([0.03158295, 0.11591389, 0.19587278, 0.09679906, 0.06501664,
       0.        , 0.08602684, 0.15786582, 0.16294774, 0.16665168,
       0.09424934, 0.0732974 , 0.00442381, 0.00500008, 0.14607594,
       0.        , 0.06326177, 0.04072143, 0.04831788, 0.12041731,
       0.07738695, 0.03234598, 0.19665775, 0.24608301, 0.16878589,
       0.06585928, 0.25658851, 0.43727752, 0.1667067 , 0.10812543,
       0.14387909, 0.09609652, 0.07786236, 0.03832289, 0.10839156,
       0.01198117, 0.22360321, 0.14818672, 0.08761724, 0.05083016,
       0.00213399, 0.0859352 , 0.0815631 , 0.13917469, 0.15265168,
       0.14914931, 0.        , 0.        , 0.        , 0.01923848,
       0.        , 0.12825845, 0.03071487, 0.        , 0.04828837,
       0.02631234, 0.        , 0.1994601 , 0.20621851, 0.23571488,
       0.04820221, 0.16454108, 0.28565627, 0.16336785, 0.13785017,
       0.08988282, 0.        , 0.        ]), 'jac_togg

Estimating beta values: 100%|██████████| 30/30 [00:00<00:00, 40.08it/s]

Saved: /home/dsemchin/COMIND/results/ppmi_gridsearch_results.npz
